In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

training_data = [
    ("def calculate_metrics(data):\n    return [x**2 for x in data]", 0), # Human
    ("Delve into the multi-faceted landscape of paradigms.", 1),        # AI Slop
    ("CRITICAL_ERROR: Task failed at memory offset 0x04F3", 2),         # Error Log
    ("for i in range(10):\n    print(i)", 0),                           # Human
    ("As an AI language model, I am happy to assist.", 1),              # AI Slop
    ("FATAL: Connection timed out after 5000ms.", 2),                   # Error Log
    ("This is just a sample text to test", 1),
    ("Same with this", 0)
]

words = []
for text, label in training_data:
  for word in text.lower().split():
    words.append(word)


vocabulary = list(set(words)) # Duplicates removed
VOCAB_SIZE = len(vocabulary)

# print(f"Vocabulary: {vocabulary}")
# print(f"Vocabulary size: {VOCAB_SIZE}")

In [15]:
def text_to_tensor(text):
  # To create a high dimensional tensor with exact size as our vocabulary
  tensor = torch.zeros(1, VOCAB_SIZE)

  for word in text.lower().split():
    if word in vocabulary:
      index = vocabulary.index(word)
      tensor[0][index] += 1

  return tensor

sample_tensor = text_to_tensor("CRITICAL_ERROR: Task failed")
# print(f"Sample vector shape: {sample_tensor.shape}")
# print(f"Sample vector: {sample_tensor}")

# Find the exact indices where the value is greater than 0
# matching_indices_ai = torch.nonzero(sample_tensor)

# print("AI Sentence has hits at vocabulary index positions:", matching_indices_ai[:, 1].tolist())


In [16]:
class TextClassifier(nn.Module):
  def __init__(self, input_size, output_size, hidden_size):
    super(TextClassifier, self).__init__()

    # Layer 1 takes VOCAB_SIZE and outputs hidden_size numbers
    self.lyr1 = nn.Linear(input_size, hidden_size)

    # Activation func
    self.relu = nn.ReLU()

    # Takes those hidden_size numbers and outputs output_size numbers (3 in our case)
    self.lyr2 = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    # This is how data flows in our model
    out = self.lyr1(x)
    out = self.relu(out)
    out = self.lyr2(out)

    return out


model = TextClassifier(input_size=VOCAB_SIZE, hidden_size=12, output_size=3)

# print("Yayyyy! Model is ready!!!")
# print(model)


In [17]:
# Now, let's test it
with torch.no_grad():
  raw_predictions = model(sample_tensor)

# print(raw_predictions)

categories = {2: "Error log", 0: "Human", 1: "AI Slop"}

predicted_index = torch.argmax(raw_predictions, dim=1).item()
# print(f"Predicted index: {predicted_index}")
# print(f"Model prediction: {categories[predicted_index]}")

In [18]:
# Loss function which takes raw output score and compares them with true category numbers
criterion = nn.CrossEntropyLoss()

# The optimizer: Stochastic Gredient Descent
# And lr (learning rate) controls how big of a jump we take while changing the parameters
optimizer = optim.SGD(model.parameters(), lr=0.1)



In [26]:
print("========== Starting out the training now ======")

for epoch in range(1, 10001):
  total_loss = 0
  for text, label in training_data:
    # Resetting model gradients so that models learns new patterns only instead of
    # including the old ones too, which would have had led it to wrong direction
    optimizer.zero_grad()

    # Converting raw text into tensor and target into python tensors,
    # to calculate loss later
    input_tensor = text_to_tensor(text)
    target_tensor = torch.tensor([label], dtype=torch.long)

    # Taking the output from the model
    output = model(input_tensor)

    # Calculating loss
    loss = criterion(output, target_tensor)
    total_loss += loss.item()

    # Calulate the direction of loss
    loss.backward()

    # Change weights to minimize loss
    optimizer.step()

  # if epoch % 10 == 0:
    # print(f"Epoch: [{epoch}/10000], Total Loss: {total_loss}")

print("========== Training done! ================")
torch.save(model.state_dict(), "text_classifier.pt")
print("========== Model saved! ================")

========== Starting out the training now ======
========== Training done! ================
========== Model saved! ================


In [25]:
print("========== Now testing the model with brand new data ================")

test_phrases = [
    "ERROR: Failed to allocate tensor memory",
    "Let us delve deeper into this multi-faceted optimization paradigm"
]

model.eval() # Switching to evaluation mode
with torch.no_grad():
  for phrase in test_phrases:
    test_tensor = text_to_tensor(phrase)

    raw_output = model(test_tensor)

    predicted_index = torch.argmax(raw_output, dim=1).item()

    print(f"\nTest phrase: {phrase}")
    print(f"Prediction: {categories[predicted_index]}")

========== Now testing the model with brand new data ================

Test phrase: ERROR: Failed to allocate tensor memory
Prediction: Error log

Test phrase: Let us delve deeper into this multi-faceted optimization paradigm
Prediction: AI Slop
